# Préparation des données

Ce notebook applique les décisions de préparation de l'EDA et vérifie le résultat. Le prétraitement lui-même est dans `src/preparation.py`, pour que le train, le test et l'application partagent le même code.

**La correction de la Tanzanie suppose vraie notre hypothèse d'inversion des libellés** (EDA, sections 3.7 et 3.8). Aucune documentation du jeu de données ne la confirme.

| Décision | Application |
|---|---|
| Lieu et statut marital de Tanzanie | Corrigés, sous notre hypothèse |
| `uniqueid` et `year` | Retirés |
| `Dont know`, `Other/Dont know/RTA`, `Other non-relatives` | Regroupés avec `Single/Never Married`, `No formal education` et `Other relative` |
| `Dont Know/Refuse to answer` | Gardé comme non-réponse |
| `Tertiary` et `Vocational` en Tanzanie | Non recodés |
| Valeurs extrêmes et lignes contradictoires | Gardées |
| Valeurs manquantes | Aucune, pas d'imputation |

In [1]:
import sys

import pandas as pd

sys.path.insert(0, "..")
from src.preparation import MODALITES_RARES, preparer

pd.set_option("display.max_columns", 50)

train_brut = pd.read_csv("../data/original/Train.csv")
test_brut = pd.read_csv("../data/original/Test.csv")

train = preparer(train_brut)
test = preparer(test_brut)

pd.DataFrame({"lignes avant": [len(train_brut), len(test_brut)], "lignes après": [len(train), len(test)],
              "colonnes avant": [train_brut.shape[1], test_brut.shape[1]], "colonnes après": [train.shape[1], test.shape[1]]},
             index=["train", "test"])

,lignes avant,lignes après,colonnes avant,colonnes après
train,23524,23524,13,11
test,10086,10086,12,10


## Vérifications

Aucune ligne n'est perdue, et les deux colonnes sont retirées. La cellule suivante s'arrête si le prétraitement crée une valeur manquante, touche un autre pays que la Tanzanie, laisse une modalité rare ou fait diverger le train et le test.

In [2]:
autres_pays = train_brut["country"] != "Tanzania"

assert len(train) == len(train_brut) and len(test) == len(test_brut)
assert not {"uniqueid", "year"} & set(train.columns) | set(test.columns) & {"uniqueid", "year"}
assert train.isna().sum().sum() == 0 and test.isna().sum().sum() == 0
for variable in ["location_type", "marital_status"]:
    regroupe = train_brut[variable].replace(MODALITES_RARES.get(variable, {}))
    assert (train.loc[autres_pays, variable] == regroupe[autres_pays]).all()
for variable, regroupement in MODALITES_RARES.items():
    assert not set(regroupement) & set(train[variable]) | set(regroupement) & set(test[variable])
for variable in test.select_dtypes(exclude="number").columns:
    assert set(test[variable].unique()) <= set(train[variable].unique())

print("Toutes les vérifications passent.")
print("Lignes de Tanzanie au lieu inversé :", (train_brut["country"] == "Tanzania").sum())
pd.crosstab(train_brut.loc[~autres_pays, "marital_status"], train.loc[~autres_pays, "marital_status"])

Toutes les vérifications passent.
Lignes de Tanzanie au lieu inversé : 6620


marital_status,Divorced/Seperated,Married/Living together,Single/Never Married,Widowed
marital_status,,,,
Divorced/Seperated,0,0,1081,0
Married/Living together,667,0,0,0
Single/Never Married,0,4229,0,0
Widowed,0,0,0,643


Le tableau croise le statut brut, en ligne, et le statut préparé, en colonne, pour la Tanzanie. Chaque statut brut correspond à un seul statut préparé, sauf `Widowed`, qui ne change pas.

## Modalités retenues

Le prétraitement ne fait qu'une transformation de libellés, sans rien apprendre des données : il ne peut donc pas créer de fuite entre train et test. Les encodages et les tranches d'âge dépendent du modèle choisi : ils seront ajustés dans la modélisation, sur le train uniquement.

In [3]:
pd.concat({
    variable: train[variable].value_counts()
    for variable in ["location_type", "marital_status", "education_level", "relationship_with_head"]
}).rename("lignes").to_frame()

lignes
location_type          Rural                             17345
                       Urban                              6179
marital_status         Married/Living together           14311
                       Single/Never Married               4843
                       Widowed                            2708
                       Divorced/Seperated                 1662
education_level        Primary education                 12791
                       No formal education                4550
                       Secondary education                4223
                       Tertiary education                 1157
                       Vocational/Specialised training     803
relationship_with_head Head of Household                 12831
                       Spouse                             6520
                       Child                              2229
                       Parent                             1086
                       Other relative                      858

## Usage dans l'application

Notre hypothèse porte sur les libellés de `Train.csv` et `Test.csv`. Dans le formulaire, une personne déclare elle-même son lieu et son statut : sa réponse n'a pas à être inversée. L'application utilise donc `preparer(..., corriger=False)`, qui garde les mêmes regroupements et les mêmes colonnes.

Dans l'exemple, la personne de Tanzanie qui répond `Rural` reste `Rural`, et le modèle la compare à des lignes corrigées sous notre hypothèse.

In [4]:
formulaire = pd.DataFrame([{
    "country": "Tanzania", "location_type": "Rural", "cellphone_access": "Yes", "household_size": 4,
    "age_of_respondent": 30, "gender_of_respondent": "Female", "relationship_with_head": "Spouse",
    "marital_status": "Single/Never Married", "education_level": "Primary education", "job_type": "Self employed",
}])
preparer(formulaire, corriger=False)

,country,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Tanzania,Rural,Yes,4,30,Female,Spouse,Single/Never Married,Primary education,Self employed
